In [1]:
import numpy as np
import heapq
import ast
import yaml
import re
import random
import json

from pprint import pprint
from openai import OpenAI
from openai import AzureOpenAI
import openai
from PIL import Image
from collections import Counter, defaultdict
from tqdm import tqdm
from queue import Queue

from helpers.env_updated import *
from helpers.utils import *
from helpers.agent_maps import *
from helpers.large_envs import *
import time

In [2]:
import os
from openai import AzureOpenAI

endpoint = "https://llm-nav.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"

subscription_key = "YOUR_AZURE_OPENAI_API_KEY"
api_version = "2025-03-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

In [3]:
# with open("../../config.yaml", "r") as f:
#     config = yaml.safe_load(f)

# OPENAI_API_KEY = config['openai']['api_key']
# MODEL = config['openai']['model']
# AZURE_API_KEY = "YOUR_AZURE_OPENAI_API_KEY"
# BASE_URL = "https://llm-nav.openai.azure.com/openai/v1"
# client = OpenAI(api_key=AZURE_API_KEY, base_url=BASE_URL)

In [4]:
import ast 
seeds = []
with open('seeds/large/bldg4_seeds.txt', 'r') as f:
    seeds = [ast.literal_eval(line) for line in f if line.strip()]

In [5]:
# openai.api_type = "azure"
# openai.api_key = "YOUR_AZURE_OPENAI_API_KEY"
# openai.azure_endpoint = "https://llm-nav.openai.azure.com/"
# openai.api_version = "2023-05-15"
# bedrock_api = "YOUR_AWS_BEDROCK_API_KEY"

In [6]:
env = Bldg4()
occupancy_grid = env.occupancy_grid
semantic_grid = env.semantic_grid

start = (27, 36)
goal = '138'
k = 30
path_steps = 3

seen_occupancy = SeenOccupancyGrid(occupancy_grid)
seen_semantic = SeenSemanticGrid(semantic_grid)
confidence_grid = ConfidenceGrid(len(occupancy_grid), len(occupancy_grid[0]))

In [7]:
from pydantic import BaseModel, Field
from typing import Literal

class NavDecision(BaseModel):
    reasoning: str = Field(description="Why this region was chosen")
    region: Literal["left", "right", "up", "down"]
    pattern: str = Field(description="Patterns in room labels you see")

def query_llm(seen_occupancy_grid, seen_semantic_grid, agent_pos, goal):
    """
    """
    with open("prompts/ours_new.txt", "r") as f:
        prompt_template = f.read()

    safe = prompt_template.replace("{", "{{").replace("}", "}}")
    for key in ["goal", "agent_pos", "seen_semantic_grid"]:
        safe = safe.replace("{{" + key + "}}", "{" + key + "}")
    
    prompt = safe.format(goal=goal, agent_pos=agent_pos, seen_occupancy_grid=seen_occupancy_grid, seen_semantic_grid=seen_semantic_grid)

    response = client.responses.parse(
        model=MODEL,
        input=[{"role": "user", "content": prompt}],
        text_format=NavDecision
    )

    return response.output_parsed

In [8]:
def run_ours_agent(seen_occupancy, seen_semantic, start, goal, timeout=250):
    steps = 0
    found_goal = False
    agent_pos = start
    total_steps = 0

    while steps < timeout:
        # print("sleeping")
        # time.sleep(15)
        steps += 1
        print(f"\nSTEP {steps}:")
        print("Agent position:", agent_pos)
        seen_occupancy.update_with_slice(agent_pos, k)
        seen_semantic.update_with_slice(agent_pos, k)
        print(seen_semantic.get_slice(agent_pos, k))
        print(seen_occupancy.mark_grid(agent_pos))

        goal_pos = seen_semantic.find_label(goal)
        if goal_pos:
            found_goal = True
            break

        if seen_occupancy.is_fully_explored():
            print("No valid path to goal.")
            break

        # query LLM
        llm_output = query_llm(seen_occupancy.get_grid(), seen_semantic.get_grid(), agent_pos, goal)
        reasoning, region, pattern = llm_output.reasoning, llm_output.region, llm_output.pattern
        print(f"Reasoning: {reasoning}")
        print(f"Region: {region}")
        print(f"Pattern: {pattern}")
        seen_semantic.update_key_with_pattern(agent_pos, "pattern", pattern)
        confidence_grid.update_frequency(agent_pos, region)
        confidence_grid.update_confidence(seen_semantic.get_grid(), goal)
        print(confidence_grid.get_grid())
        max_confidence_pos = confidence_grid.find_max_confidence_pos()

        # next step
        path_to_max_confidence = seen_occupancy.plan_towards(agent_pos, max_confidence_pos)
        if len(path_to_max_confidence) < path_steps:
            agent_pos = path_to_max_confidence[-1]
            total_steps += len(path_to_max_confidence)
        else:
            try:
                agent_pos = path_to_max_confidence[path_steps] # set agent's next position
                total_steps += path_steps
            except:
                print("No valid path to max confidence.")
                break


    if found_goal:
        goal_pos = seen_semantic.find_label(goal)
        path = seen_occupancy.astar(agent_pos, goal_pos)
        if path:
            print(path)
            total_steps += len(path) - 2
            print(f"Found goal in {total_steps} steps.")
            return total_steps
        else:
            return total_steps

    else:
        print("No goal found")
        return -1

In [9]:
with open('seeds/large/bldg4_ours.txt', 'w') as f:
    i = 5
    print(f"SEED {i}")
    result = run_ours_agent(seen_occupancy, seen_semantic, seeds[i]['start_pos'], seeds[i]['target_room'], timeout=250)
    f.write(str(result))
    f.write('\n')
    print(f"Path {seeds[i]['start_pos']} -> {seeds[i]['target_pos']}: {result}")
    print(f"Path length: {result}") 

SEED 5

STEP 1:
Agent position: (19, 118)
[[{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}], [{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}], [{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}], [{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}], [{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}], [{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}], [{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}], [{}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}

NameError: name 'MODEL' is not defined